### Problem

You are given a table containing customer purchases.

For each customer and each month:

- Calculate the monthly spending.
- Show the previous month's spending.
- Calculate the month-over-month growth percentage.
- If there is no previous calendar month (for example January → March because February is missing), then the previous month's spending should be 0.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customer_orders AS
SELECT *
FROM VALUES
('C1', '2026-01-02', 200),
('C1', '2026-01-15', 300),
('C1', '2026-03-10', 800),
('C1', '2026-03-15', 200),

('C2', '2026-01-05', 400),
('C2', '2026-02-10', 600),
('C2', '2026-03-08', 750),
('C2', '2026-03-20', 250),

('C3', '2026-02-01', 500),
('C3', '2026-04-15', 700)

AS customer_orders(customer_id, order_date, amount);

In [0]:

%sql
select date(DATE_TRUNC('month', order_date)) as start_month , * from customer_orders
    


In [0]:
%sql

with RECURSIVE month_rollup as (select customer_id, date(DATE_TRUNC('month', order_date)) as start_month, sum(amount) as amt from customer_orders group by all ) 
select * from month_rollup

In [0]:

%sql

with RECURSIVE month_rollup as (select customer_id, date(DATE_TRUNC('month', order_date)) as start_month, sum(amount) as amt from customer_orders group by all ),

calender_dates as (
select min (start_month) as cal_months from month_rollup
union all
select cal_months  + INTERVAL '1' MONTH from calender_dates
where cal_months < (select max(start_month) from month_rollup)

),

customer_month as (
select distinct customer_id,cal_months  from month_rollup m
cross join calender_dates   
),

missing_month_rollup as (
select m.customer_id, m.cal_months, coalesce(r.amt,0) as amount from customer_month m
left join month_rollup r 
on m.customer_id = r.customer_id and m.cal_months = r.start_month
),
prev_month as (
select customer_id, cal_months,amount, lag(amount) over (partition by customer_id order by cal_months) as prev_month_amt from missing_month_rollup
)

select * from (
select *, coalesce(((amount - prev_month_amt)/nullif(prev_month_amt,0))*100,100) as growth from prev_month order by customer_id, cal_months ) where growth > 25
    


### A Sale price increases for several months.
### 
Find the month where the trend changed.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW monthly_sales AS
SELECT *
FROM VALUES
(1, 'Jan', 100),
(2, 'Feb', 120),
(3, 'Mar', 140),
(4, 'Apr', 130),
(5, 'May', 110),
(6, 'Jun', 115),
(7, 'Jul', 120)

AS monthly_sales(month_id, month, sales);

In [0]:
%sql
with sale_values as (select month_id, month, sales, lag(sales) over (order by month_id asc ) as prev_sale from monthly_sales),
change_trend (
select *, 
case when 
(sales - coalesce(prev_sale,0)) > 0 then 'Increase_Trend'
when (sales - prev_sale) < 0  then 'Decrease_Trend'
else 'No_Change_Trend'  end as change_sale

 from sale_values
),
flag_cal (

    select *, case when lag(change_sale) over (order by month_id) = change_sale then 'No_Changed_Trend' else 'Changed_Trend' end as flag from  change_trend
)
select * from flag_cal

